# BiLSTM Seq2Seq — Akkadian → English Machine Translation
**Baseline model** with encoder-decoder architecture, teacher forcing, and evaluation via BLEU + chrF++.

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.model_selection import train_test_split
from sacrebleu import corpus_bleu, corpus_chrf
from tqdm import tqdm
import random
import numpy as np

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
# torch.backends.cudnn.deterministic = True
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## 1. Load & Clean Data

In [2]:
df = pd.read_csv("data.csv")
# df = df[['source', 'translation']].dropna()
df = df[['transliteration', 'translation']].dropna()
df = df.rename(columns={'transliteration': 'source'})

def clean_text(text):
    text = text.lower()
    text = text.replace('"', '')
    text = text.replace('<gap>', ' <sep> ')
    return text

df['source'] = df['source'].apply(clean_text)
df['translation'] = df['translation'].apply(clean_text)

# Add special tokens to target
df['translation'] = "<sos> " + df['translation'] + " <eos>"

print(f"Total samples: {len(df)}")
print("\nSample source:", df.iloc[0]['source'])
print("Sample target:", df.iloc[0]['translation'])

Total samples: 62196

Sample source: um-ma en-na-sú-in-ma a-na en-um-a-šùr qí-bi4-ma a-hi a-ta a-ma-kam tí-ir-tí a-ṣé-er i-tur4-dingir i-li-kam a-dí a-wa-tim ša ma-šu-ha-ku-ni ù a-ta a-na i-tur4-dingir pu-nu-ma ú i-na ša-ha-tí-šu i-zi-iz-ma šu-ma a-wa-tum i-sí-ú i-zi  <sep>  šu-ma lá i- <sep> -ší  <sep>  me-et ni  <sep>  me  <sep>  i- <sep>  um-ma  <sep>  le-qé-a-ma  <sep>  lá a-na  <sep>  ba-lá-ṭí  <sep>  i  <sep>  gi-mì-lam  <sep>  i-na ṣé-ri-a šu-ku-un
Sample target: <sos> from enna-suen to ennam-aššur: my dear brother, my message came to itūr-ilī there concerning the matter where i have been treated badly. you too should turn to itūr-ilī and assist him, and if they have called a lawsuit  <sep>  do me a great favour. <eos>


In [3]:
# token lengths
df['src_len'] = df['source'].apply(lambda x: len(x.split()))
df['tgt_len'] = df['translation'].apply(lambda x: len(x.split()))

def print_stats(name, series):
    print(f"\n{name} stats:")
    print(f"Min: {series.min()}")
    print(f"Max: {series.max()}")
    print(f"Mean: {series.mean():.2f}")
    print(f"Median: {series.median()}")

print_stats("Source (Akkadian)", df['src_len'])
print_stats("Target (English)", df['tgt_len'])


Source (Akkadian) stats:
Min: 1
Max: 784
Mean: 21.50
Median: 9.0

Target (English) stats:
Min: 3
Max: 749
Mean: 30.44
Median: 14.0


In [4]:
MAX_LEN = 50

df = df[df['src_len'] <= MAX_LEN]
df = df[df['tgt_len'] <= MAX_LEN]

## 2. Train / Validation / Test Split

Split **before** building vocabulary to avoid data leakage.  
Split: 80% train, 10% validation, 10% test.

In [5]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED)

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 40998 | Val: 5125 | Test: 5125


## 3. Tokenisation & Vocabulary

Vocabulary is built **only from the training set**.  
`<pad>`, `<unk>`, `<sos>`, `<eos>` are always reserved regardless of frequency.

In [6]:
def tokenize(text):
    return text.split()

for split_df in [train_df, val_df, test_df]:
    split_df['src_tokens'] = split_df['source'].apply(tokenize)
    split_df['tgt_tokens'] = split_df['translation'].apply(tokenize)

def build_vocab(token_lists, min_freq=2):
    """
    Build vocabulary from token lists.
    Special tokens are always included at fixed indices:
      0: <pad>, 1: <unk>, 2: <sos>, 3: <eos>
    """
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)

    # Reserve special tokens at fixed positions
    vocab = {"<pad>": 0, "<unk>": 1, "<sos>": 2, "<eos>": 3}

    for word, freq in counter.items():
        if freq >= min_freq and word not in vocab:
            vocab[word] = len(vocab)

    return vocab

# Build vocab from TRAIN only
src_vocab = build_vocab(train_df['src_tokens'])
tgt_vocab = build_vocab(train_df['tgt_tokens'])
tgt_inv_vocab = {v: k for k, v in tgt_vocab.items()}

print(f"Source vocab size: {len(src_vocab)}")
print(f"Target vocab size: {len(tgt_vocab)}")

Source vocab size: 37305
Target vocab size: 17527


In [7]:
def encode(tokens, vocab):
    return [vocab.get(t, vocab["<unk>"]) for t in tokens]

for split_df in [train_df, val_df, test_df]:
    split_df['src_ids'] = split_df['src_tokens'].apply(lambda x: encode(x, src_vocab))
    split_df['tgt_ids'] = split_df['tgt_tokens'].apply(lambda x: encode(x, tgt_vocab))

## 4. Dataset & DataLoader

In [8]:
class TranslationDataset(Dataset):
    def __init__(self, df):
        self.src = df['src_ids'].tolist()
        self.tgt = df['tgt_ids'].tolist()

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return torch.tensor(self.src[idx]), torch.tensor(self.tgt[idx])


def collate_fn(batch):
    src, tgt = zip(*batch)
    src = pad_sequence(src, padding_value=0)  # (src_len, batch)
    tgt = pad_sequence(tgt, padding_value=0)  # (tgt_len, batch)
    return src, tgt


train_dataset = TranslationDataset(train_df)
val_dataset   = TranslationDataset(val_df)
test_dataset  = TranslationDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

Train batches: 1282 | Val batches: 161 | Test batches: 161


## 5. Model — BiLSTM Encoder / LSTM Decoder

- **Encoder**: bidirectional LSTM; forward+backward hidden/cell states are concatenated and projected to decoder hidden size.
- **Decoder**: unidirectional LSTM with teacher forcing.
- **Dropout** added to both encoder and decoder embeddings + LSTM for regularisation.

In [9]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=0)
        self.dropout   = nn.Dropout(dropout)
        self.lstm      = nn.LSTM(emb_dim, hid_dim, bidirectional=True, batch_first=False)
        self.fc_hidden = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_cell   = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, src):
        # src: (src_len, batch)
        embedded = self.dropout(self.embedding(src))  # (src_len, batch, emb_dim)
        outputs, (hidden, cell) = self.lstm(embedded)
        # Concatenate last forward and backward hidden/cell states
        hidden = torch.tanh(self.fc_hidden(torch.cat((hidden[-2], hidden[-1]), dim=1))).unsqueeze(0)
        cell   = torch.tanh(self.fc_cell(  torch.cat((cell[-2],   cell[-1]),   dim=1))).unsqueeze(0)
        return hidden, cell


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=0)
        self.dropout   = nn.Dropout(dropout)
        self.lstm      = nn.LSTM(emb_dim, hid_dim)
        self.fc        = nn.Linear(hid_dim, output_dim)

    def forward(self, input, hidden, cell):
        # input: (batch,)  →  unsqueeze to (1, batch)
        input    = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))          # (1, batch, emb_dim)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc(output.squeeze(0))                 # (batch, output_dim)
        return prediction, hidden, cell


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = tgt.shape[1]
        tgt_len    = tgt.shape[0]
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(tgt_len, batch_size, vocab_size).to(self.device)
        hidden, cell = self.encoder(src)

        input = tgt[0]  # <sos> token
        for t in range(1, tgt_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            top1  = output.argmax(1)
            input = tgt[t] if torch.rand(1).item() < teacher_forcing_ratio else top1

        return outputs

## 6. Initialise Model, Optimiser & Loss

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

EMB_DIM = 256
HID_DIM = 512
DROPOUT = 0.5

encoder = Encoder(len(src_vocab), EMB_DIM, HID_DIM, DROPOUT)
decoder = Decoder(len(tgt_vocab), EMB_DIM, HID_DIM, DROPOUT)
model   = Seq2Seq(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore <pad>

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

Using device: cuda
Trainable parameters: 28,808,823


## 7. Inference Helper

In [11]:
def translate_sentence(model, sentence, src_vocab, tgt_inv_vocab, device, max_len=50):
    """Greedy decode a single source sentence to English."""
    model.eval()
    tokens     = sentence.split()
    src_ids    = [src_vocab.get(t, src_vocab["<unk>"]) for t in tokens]
    src_tensor = torch.tensor(src_ids).unsqueeze(1).to(device)  # (src_len, 1)

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    input_token = torch.tensor([tgt_vocab["<sos>"]]).to(device)
    outputs     = []

    for _ in range(max_len):
        with torch.no_grad():
            output, hidden, cell = model.decoder(input_token, hidden, cell)
        pred_token = output.argmax(1).item()
        word       = tgt_inv_vocab.get(pred_token, "<unk>")
        if word == "<eos>":
            break
        outputs.append(word)
        input_token = torch.tensor([pred_token]).to(device)

    return " ".join(outputs)

## 8. Evaluation Function

Runs on a full DataFrame split and returns BLEU and chrF++ scores.

In [12]:
def evaluate_model(model, df_split, src_vocab, tgt_vocab, tgt_inv_vocab, device):
    """
    Evaluate model on a dataset split.
    Returns BLEU, chrF++ (sacrebleu corpus-level scores).
    """
    preds = []
    refs  = []

    for _, row in df_split.iterrows():
        src = row['source']
        # Strip <sos>/<eos> to get the clean reference
        ref = row['translation'].replace("<sos> ", "").replace(" <eos>", "")

        pred = translate_sentence(model, src, src_vocab, tgt_inv_vocab, device)

        # Normalise <sep> marker consistently in both pred and ref
        pred = pred.replace("<sep>", " ").strip()
        ref  = ref.replace("<sep>", " ").strip()

        preds.append(pred)
        refs.append(ref)

    bleu = corpus_bleu(preds, [refs]).score
    chrf = corpus_chrf(preds, [refs]).score  # chrF++ (word_order=2 by default in sacrebleu)

    return bleu, chrf

## 9. Training Loop

- Gradient clipping (`max_norm=1.0`) to prevent exploding gradients
- Validation loss tracked every epoch
- Best model checkpoint saved based on **validation loss**
- BLEU + chrF++ logged on validation set every epoch

In [13]:
N_EPOCHS   = 100
CLIP       = 1.0
PATIENCE = 10
patience_counter = 0
BEST_PATH  = "bilstm_best.pt"

best_val_loss = float('inf')
history = []

for epoch in range(N_EPOCHS):

    # ── Training ──────────────────────────────────────────────
    model.train()
    train_loss = 0

    for src, tgt in tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS} [Train]"):
        src, tgt = src.to(device), tgt.to(device)

        optimizer.zero_grad()
        output     = model(src, tgt)                     # (tgt_len, batch, vocab)
        output_dim = output.shape[-1]

        # Flatten, skipping the <sos> at position 0
        output = output[1:].view(-1, output_dim)
        tgt    = tgt[1:].reshape(-1)

        loss = criterion(output, tgt)
        loss.backward()

        # Gradient clipping — prevents exploding gradients in LSTMs
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)

        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ── Validation loss ───────────────────────────────────────
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt   = src.to(device), tgt.to(device)
            output     = model(src, tgt, teacher_forcing_ratio=0.0)  # no teacher forcing at eval
            output_dim = output.shape[-1]
            output     = output[1:].view(-1, output_dim)
            tgt_flat   = tgt[1:].reshape(-1)
            val_loss  += criterion(output, tgt_flat).item()

    avg_val_loss = val_loss / len(val_loader)

    # ── BLEU / chrF++ on validation set ───────────────────────
    val_bleu, val_chrf = evaluate_model(
        model, val_df, src_vocab, tgt_vocab, tgt_inv_vocab, device
    )

    history.append({
        'epoch': epoch + 1,
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss,
        'val_bleu': val_bleu,
        'val_chrf': val_chrf
    })

    print(f"\nEpoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} "
          f"| Val BLEU: {val_bleu:.2f} | Val chrF++: {val_chrf:.2f}")

    # ── Checkpoint best model ─────────────────────────────────
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), BEST_PATH)
        patience_counter = 0
        print(f"  ✅ New best model saved (val_loss={best_val_loss:.4f})")
    else:
        patience_counter += 1
        print(f"  ⏳ No improvement ({patience_counter}/{PATIENCE})")

        if patience_counter >= PATIENCE:
            print("🛑 Early stopping triggered")
            break

print("\nTraining complete.")

Epoch 1/100 [Train]: 100%|██████████| 1282/1282 [04:55<00:00,  4.33it/s]



Epoch 1 | Train Loss: 5.9581 | Val Loss: 6.0679 | Val BLEU: 0.04 | Val chrF++: 4.62
  ✅ New best model saved (val_loss=6.0679)


Epoch 2/100 [Train]: 100%|██████████| 1282/1282 [04:46<00:00,  4.48it/s]



Epoch 2 | Train Loss: 5.4713 | Val Loss: 6.0742 | Val BLEU: 0.03 | Val chrF++: 5.30
  ⏳ No improvement (1/10)


Epoch 3/100 [Train]: 100%|██████████| 1282/1282 [04:46<00:00,  4.47it/s]


...


Epoch 59 | Train Loss: 2.7503 | Val Loss: 5.3107 | Val BLEU: 6.28 | Val chrF++: 21.68
  ⏳ No improvement (8/10)


Epoch 60/100 [Train]: 100%|██████████| 1282/1282 [04:45<00:00,  4.49it/s]



Epoch 60 | Train Loss: 2.7345 | Val Loss: 5.3031 | Val BLEU: 5.60 | Val chrF++: 21.20
  ⏳ No improvement (9/10)


Epoch 61/100 [Train]: 100%|██████████| 1282/1282 [04:46<00:00,  4.48it/s]



Epoch 61 | Train Loss: 2.7175 | Val Loss: 5.3024 | Val BLEU: 4.39 | Val chrF++: 20.79
  ⏳ No improvement (10/10)
🛑 Early stopping triggered

Training complete.


## 10. Final Evaluation on Test Set

Load the **best checkpoint** and evaluate on the held-out test set.

In [14]:
# Load best model weights
model.load_state_dict(torch.load(BEST_PATH, map_location=device))

test_bleu, test_chrf = evaluate_model(
    model, test_df, src_vocab, tgt_vocab, tgt_inv_vocab, device
)

print("=" * 40)
print("  TEST SET RESULTS (BiLSTM Baseline)")
print("=" * 40)
print(f"  BLEU   : {test_bleu:.2f}")
print(f"  chrF++ : {test_chrf:.2f}")
print("=" * 40)

C:\Users\surya\AppData\Local\Temp\ipykernel_41616\1897487650.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(BEST_PATH, map_location=dev

  TEST SET RESULTS (BiLSTM Baseline)
  BLEU   : 6.01
  chrF++ : 20.04


## 11. Training History

In [15]:
history_df = pd.DataFrame(history)
print(history_df.to_string(index=False))

 epoch  train_loss  val_loss  val_bleu  val_chrf
     1    5.958099  6.067881  0.039616  4.623674
     2    5.471324  6.074204  0.033304  5.304828
     3    5.255387  6.047581  0.009728  4.351366
     4    5.128945  6.054918  0.012259  4.352733
     5    4.999298  6.047806  0.027731  4.887837
     6    4.916989  6.033161  0.025957  5.293604
     7    4.847654  6.202116  0.030162  3.994516
     8    4.800441  5.990773  0.029998  5.269145
     9    4.721707  6.101379  0.058349  5.378075
    10    4.667882  6.160084  0.254201  7.650594
    11    4.619483  6.052672  0.808736  8.659951
    12    4.527987  6.054433  0.737636  9.470532
    13    4.437210  5.926950  1.218293 10.188375
    14    4.347411  5.902427  0.665317 10.183380
    15    4.239641  5.881556  1.546996 11.871176
    16    4.180306  5.775846  1.125994 11.616895
    17    4.107501  5.853635  1.190795 12.518199
    18    4.050567  5.754735  1.614699 12.846337
    19    3.986018  5.804009  1.376950 13.019119
    20    3.916227  